# Biohub Cell Tracking — PyTorch Training Notebook

This notebook trains the 3D U-Net model for cell detection and movement vector prediction on the zebrafish embryo cells dataset.

### Features:
- **Resume-aware**: Automatically searches for `last.pt`, `best.pt`, and `training_log.csv` in any attached dataset directories (using robust recursive search) and resumes seamlessly.
- **Log Continuity**: Appends training indicators to the loaded log history, ensuring history is not lost across run restarts.
- **Epoch-Level Outputs**: Saves `last.pt`, `best.pt`, and updates `training_log.csv` at the end of every epoch.
- **Live Plotting**: Generates and saves a training/validation curve graph (`loss_plot.png`) after every epoch and displays it in the notebook output.

In [1]:
# ── Install/upgrade dependencies ──
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('zarr>=3.0', 'scipy>=1.10', 'pandas>=2.0', 'tqdm', 'pyyaml', 'matplotlib', 'scikit-image>=0.21')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 78.7 MB/s eta 0:00:00


In [2]:
# ── Environment & Path Setup ──
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from IPython.display import clear_output, display

# ── YOUR UPLOADED DATASET NAME ──
# Change this single variable whenever you upload under a new username/dataset name:
USER_DATASET_NAME = "neutrinomilan/biohub-cell-tracking-src"

# ── FIXED PATHS ──
COMPETITION_DATA_ROOT = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
USER_DATASET_ROOT = Path(f"/kaggle/input/datasets/{USER_DATASET_NAME}")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add source directory to Python path instantly
for p in [USER_DATASET_ROOT, USER_DATASET_ROOT / "src", OUTPUT_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

INPUT_CHECKPOINT_DIRS = [USER_DATASET_ROOT, USER_DATASET_ROOT / "src", OUTPUT_DIR]

last_ckpt_path = OUTPUT_DIR / "last.pt"
best_ckpt_path = OUTPUT_DIR / "best.pt"
log_csv_path = OUTPUT_DIR / "training_log.csv"
plot_png_path = OUTPUT_DIR / "loss_plot.png"

print(f"Competition Data: {COMPETITION_DATA_ROOT}")
print(f"User Dataset Root: {USER_DATASET_ROOT}")
print("Paths configured instantly.")


COMPETITION_DATA_ROOT auto-resolved to: /kaggle/input/competitions/biohub-cell-tracking-during-development/train
Paths configured.


In [3]:
# ── Config Setup ──
from src.config import Config, ModelConfig, DataConfig, TrainingConfig
from src.models.residual_unet3d import ResidualCellTrackingUNet

# Choose model architecture: ResidualCellTrackingUNet
MODEL_CLASS = ResidualCellTrackingUNet 

# Initialize configuration
config = Config(
    model=ModelConfig(
        in_channels=1,
        base_features=16,
        feature_multiplier=3.0,
        downsample_factors=[[1, 2, 2], [2, 2, 2], [2, 2, 2]],
        nms_kernel=3,
        nms_threshold=0.5
    ),
    data=DataConfig(
        data_root=COMPETITION_DATA_ROOT,
        patch_size=[64, 128, 128],
        train_full_volume=False,
        gaussian_sigma=[0.5, 2.0, 2.0],
        cell_mask_radius=1,
        scale=[1.625, 0.40625, 0.40625],
        normalize_mode="percentile",
        reject_empty=True,
        center_on_cell_prob=0.5,
        input_shape=[64, 128, 128]
    ),
    training=TrainingConfig(
        lr=2.5e-5,
        batch_size=4,
        gradient_accumulation_steps=2,
        epochs=150,
        amp=True,
        swa_start_epoch=80,
        val_every=250,
        w_bg=1.0,
        w_fg=1.0,
        lambda_mv=1.0,
        vector_mask_mode="curriculum",
        vector_curriculum_epoch=40,
        checkpoint_dir=str(OUTPUT_DIR),
        num_workers=4,
        val_embryo_fraction=0.25,
        seed=42,
        weight_decay=1e-4,
        patience=20
    )
)

print("Config loaded successfully.")


Config loaded successfully.


In [4]:
# ── Checkpoint & Log History Resolution ──
import os
import zipfile
import pandas as pd
import torch
from pathlib import Path

def find_file_fast(filename, folders, is_dir=False):
    resolved_folders = []
    for f in folders:
        resolved_folders.append(f)
        comp_dir = f / "competitions"
        if comp_dir.exists():
            resolved_folders.append(comp_dir)
            
    for folder in resolved_folders:
        if not folder.exists():
            continue
        if folder == Path("/kaggle/working"):
            target = folder / filename
            if is_dir and target.is_dir(): return target
            if not is_dir and target.is_file(): return target
        else:
            for child in folder.iterdir():
                if "biohub-cell-tracking-during-development" in child.name:
                    continue
                if is_dir and child.is_dir() and child.name == filename: return child
                if not is_dir and child.is_file() and child.name == filename: return child
                if child.is_dir():
                    for p in child.rglob(filename):
                        if is_dir and p.is_dir(): return p
                        if not is_dir and p.is_file(): return p
    return None

def reconstruct_pt(folder_path, target_path):
    import zipfile, os
    print(f"\n[!] Found unzipped checkpoint folder at {folder_path}")
    true_root = folder_path
    for p in folder_path.rglob("data.pkl"):
        true_root = p.parent
        break
    print(f"    Repacking it into strict PyTorch format at {target_path}...")
    with zipfile.ZipFile(target_path, 'w', compression=zipfile.ZIP_STORED) as zf:
        for root, _, files in os.walk(true_root):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, true_root).replace('\\', '/')
                arcname = "archive/" + arcname
                zf.write(file_path, arcname)
    return target_path

# Check for unzipped Kaggle 'last' or 'best' folders first (ALWAYS prioritizing 'last' folder if present!)
last_folder = find_file_fast("last", INPUT_CHECKPOINT_DIRS, is_dir=True)
best_folder = find_file_fast("best", INPUT_CHECKPOINT_DIRS, is_dir=True)

if last_folder and last_folder.is_dir():
    input_last = reconstruct_pt(last_folder, OUTPUT_DIR / "last.pt")
else:
    input_last = find_file_fast("last.pt", INPUT_CHECKPOINT_DIRS)

if best_folder and best_folder.is_dir():
    input_best = reconstruct_pt(best_folder, OUTPUT_DIR / "best.pt")
else:
    input_best = find_file_fast("best.pt", INPUT_CHECKPOINT_DIRS)

input_log = find_file_fast("training_log.csv", INPUT_CHECKPOINT_DIRS)

resume_ckpt = None
log_history = pd.DataFrame(columns=['epoch', 'train_loss', 'val_loss', 'val_loss_ci', 'val_loss_mv', 'lr'])
start_epoch = 0
best_val_loss = float('inf')
val_checks_without_improvement = 0

# Resolve last.pt
if input_last and input_last.exists():
    print(f"Found last.pt at: {input_last}")
    try:
        resume_ckpt = torch.load(input_last, map_location='cpu')
        val_checks_without_improvement = resume_ckpt.get('val_checks_without_improvement', 0)
        print(f"\n{'='*50}")
        print(f"✅ SUCCESS: Model successfully loaded from checkpoint!")
        print(f"   -> Checkpoint stopped at Epoch: {resume_ckpt.get('epoch', 'Unknown')}")
        if 'val_loss' in resume_ckpt:
            print(f"   -> Checkpoint recorded Val Loss: {resume_ckpt['val_loss']:.4f}")
        print(f"{'='*50}\n")
    except Exception as e:
        print(f"Failed to load last.pt: {e}")

# Resolve training_log.csv
if input_log and input_log.exists():
    print(f"Found training_log.csv at: {input_log}")
    try:
        log_history = pd.read_csv(input_log)
        if not log_history.empty and 'epoch' in log_history.columns:
            start_epoch = int(log_history['epoch'].max())
            print(f"Continuing from completed Epoch {start_epoch} (Starting Epoch {start_epoch + 1})")
            if 'val_loss' in log_history.columns:
                best_val_loss = log_history['val_loss'].min()
                print(f"Best recorded validation loss so far: {best_val_loss:.4f}")
    except Exception as e:
        print(f"Failed to load log history: {e}")

if start_epoch == 0 and resume_ckpt is not None and 'epoch' in resume_ckpt:
    start_epoch = int(resume_ckpt['epoch'])
    print(f"Set start_epoch from loaded checkpoint: Epoch {start_epoch}")

if start_epoch == 0:
    print("No previous checkpoints resolved. Starting fresh training from Epoch 1.")


Found last.pt at: /kaggle/input/datasets/freak2209/biohub-cell-tracking-src/src/last.pt

✅ SUCCESS: Model successfully loaded from checkpoint!
   -> Checkpoint stopped at Epoch: 24

Found training_log.csv at: /kaggle/input/datasets/freak2209/biohub-cell-tracking-src/src/training_log.csv
Continuing from completed Epoch 24 (Starting Epoch 25)
Best recorded validation loss so far: 0.0218


In [5]:
# ── Training Components Initialization ──
from src.train import set_seed, build_dataloaders, validate, _update_swa_bn
from src.losses.detection_loss import DetectionLoss

# Setup reproducibility
set_seed(config.training.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Ensure model input shape matches training patch size to avoid crop size mismatch
config.data.input_shape = config.data.patch_size

# Initialize Model
model = MODEL_CLASS.from_config(config).to(device)

# --- CAPTURE SHAPE BEFORE DATAPARALLEL ---
output_shape = model.output_shape 

if torch.cuda.device_count() > 1:
    print(f"Awesome, using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

print(f"Model Output Shape: {output_shape} (Input: {config.data.input_shape})")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Loss function
loss_fn = DetectionLoss(
    lambda_mv=config.training.lambda_mv,
    vector_mask_mode=config.training.vector_mask_mode,
    curriculum_epoch=config.training.vector_curriculum_epoch,
    output_shape=output_shape,  # <-- Using captured shape
)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.training.lr,
    weight_decay=config.training.weight_decay,
)

# Schedulers
n_non_swa_epochs = max(1, config.training.swa_start_epoch)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=n_non_swa_epochs,
    eta_min=config.training.lr * 0.01,
)

# SWA
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=config.training.swa_lr if hasattr(config.training, 'swa_lr') else config.training.lr * 0.05)
swa_started = False

# AMP Scaler
use_amp = config.training.amp and device.type == "cuda"
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# Dataloaders
base_model = model.module if hasattr(model, 'module') else model
train_loader, val_loader = build_dataloaders(config, base_model)

Training on: cuda
Awesome, using 2 GPUs!
Model Output Shape: (16, 36, 36) (Input: [64, 128, 128])
Trainable parameters: 6,964,036
Dataset split: 128 train samples, 71 val samples


In [6]:
# ── Restore State dicts if Resuming ──
if resume_ckpt is not None:
    print("Restoring model state_dict...")
    
    # --- PREFIX AUTOCORRECT FOR DATAPARALLEL ---
    state_dict = resume_ckpt['model']
    if hasattr(model, 'module') and not any(k.startswith('module.') for k in state_dict.keys()):
        state_dict = {f"module.{k}": v for k, v in state_dict.items()}
    elif not hasattr(model, 'module') and any(k.startswith('module.') for k in state_dict.keys()):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    
    # Smart check: Only reset optimizer if old checkpoint lacks weight_decay
    opt_groups = resume_ckpt.get('optimizer', {}).get('param_groups', [])
    has_weight_decay = any(pg.get('weight_decay', 0) > 0 for pg in opt_groups)

    if has_weight_decay:
        print("✅ Restoring existing optimizer state_dict (weight_decay already active)...")
        optimizer.load_state_dict(resume_ckpt['optimizer'])
        # Reset LR to 1e-4 so cosine scheduler decays from the correct base.
        # Without this, the scheduler would inherit the low decayed LR (e.g.
        # 2.5e-5) from last.pt and treat that as its peak, making T_max=60
        # useless since there is nowhere to decay from.
        for pg in optimizer.param_groups:
            pg['lr'] = 1e-4
        # Always reinitialize cosine scheduler with T_max=60 so LR decays
        # slowly across multiple 3-epoch Kaggle sessions, not within one.
        cosine_scheduler = CosineAnnealingLR(
            optimizer,
            T_max=60,
            eta_min=config.training.lr * 0.01,
        )
        print(f"✅ Cosine scheduler reinitialized with T_max=60, base_lr=1e-4")
    else:
        print("🧹 Old checkpoint without weight_decay detected. Performing one-time fresh optimizer reset...")
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=1e-4,
            weight_decay=config.training.weight_decay,
        )
        # Use a fixed T_max of 60 epochs so LR decays slowly across multiple sessions
        cosine_scheduler = CosineAnnealingLR(
            optimizer,
            T_max=60,
            eta_min=config.training.lr * 0.01,
        )
        print(f"✅ One-time reset complete (lr=1e-4, weight_decay={config.training.weight_decay}). Future checkpoints will load normally.")

    if 'swa_started' in resume_ckpt:
        swa_started = resume_ckpt['swa_started']
        
    if 'swa_model' in resume_ckpt and resume_ckpt['swa_model'] is not None:
        print("Restoring SWA model state_dict...")
        swa_state = resume_ckpt['swa_model']
        
        # SWA Prefix autocorrect
        if hasattr(model, 'module') and not any(k.startswith('module.module.') for k in swa_state.keys()):
            new_swa = {}
            for k, v in swa_state.items():
                if k.startswith('module.'): new_swa[f"module.{k}"] = v
                else: new_swa[k] = v
            swa_state = new_swa
        elif not hasattr(model, 'module') and any(k.startswith('module.module.') for k in swa_state.keys()):
            new_swa = {}
            for k, v in swa_state.items():
                if k.startswith('module.module.'): new_swa[k.replace('module.', '', 1)] = v
                else: new_swa[k] = v
            swa_state = new_swa
            
        swa_model.load_state_dict(swa_state)
        
    print("State restoration complete.")

Restoring model state_dict...
Restoring optimizer state_dict...
Restoring cosine scheduler state_dict...
Restoring SWA scheduler state_dict...
Restoring SWA model state_dict...
Restoring AMP scaler state_dict...
State restoration complete.


In [7]:
# ── Helper: Live Plotting Function ──
def plot_loss_curves(log_df, save_path):
    if log_df.empty:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Total loss
    axes[0, 0].plot(log_df['epoch'], log_df['train_loss'], label='Train Loss', marker='o')
    axes[0, 0].plot(log_df['epoch'], log_df['val_loss'], label='Val Loss', marker='x')
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].legend()
    
    # CI loss
    axes[0, 1].plot(log_df['epoch'], log_df['train_loss_ci'], label='Train CI', marker='o')
    axes[0, 1].plot(log_df['epoch'], log_df['val_loss_ci'], label='Val CI', marker='x')
    axes[0, 1].set_title('Cell Indicator Loss')
    axes[0, 1].legend()
    
    # Tracking Losses
    axes[1, 0].plot(log_df['epoch'], log_df['train_loss_la'], label='Train LA (Full)', marker='o')
    axes[1, 0].plot(log_df['epoch'], log_df['train_loss_lb'], label='Train LB (Maxima)', marker='o')
    if 'val_loss_la' in log_df.columns:
        axes[1, 0].plot(log_df['epoch'], log_df['val_loss_la'], label='Val LA', marker='x')
        axes[1, 0].plot(log_df['epoch'], log_df['val_loss_lb'], label='Val LB', marker='x')
    axes[1, 0].set_title('Tracking Losses')
    axes[1, 0].legend()
    
    # Alpha
    axes[1, 1].plot(log_df['epoch'], log_df['train_alpha'], label='Curriculum Alpha', marker='o')
    axes[1, 1].set_title('Curriculum Alpha')
    axes[1, 1].legend()
    
    fig.tight_layout()  
    
    plt.savefig(save_path)
    display(fig)
    plt.close()


In [8]:
# ── Main Training Loop ──
import torch
from tqdm.notebook import tqdm
import pandas as pd
from IPython.display import clear_output

# OVERRIDE VALIDATE TO FIX PYTORCH DATAPARALLEL KWARG BUG
@torch.no_grad()
def validate(model, loss_fn, loader, device):
    model.eval()
    totals = {"loss": 0.0, "loss_ci": 0.0, "loss_mv": 0.0, "loss_la": 0.0, "loss_lb": 0.0, "alpha": 0.0}
    n = 0
    for batch in loader:
        volume  = batch["volume"].to(device)
        targets = {
            "cell_indicator":  batch["cell_indicator"].to(device),
            "cell_mask":       batch["cell_mask"].to(device),
            "movement_vectors":batch["movement_vectors"].to(device),
        }
        outputs = model(volume, False)
        losses  = loss_fn(outputs, targets)
        bs = volume.size(0)
        for k in totals:
            if k in losses: totals[k] += float(losses[k]) * bs
        n += bs
    model.train()
    return {k: v / max(n, 1) for k, v in totals.items()}

print(f"Beginning training run from Epoch {start_epoch + 1} to {config.training.epochs}...")

global_step = start_epoch * len(train_loader)

for epoch in range(start_epoch, config.training.epochs):
    loss_fn.set_epoch(epoch)
    model.train()

    if epoch >= config.training.swa_start_epoch and not swa_started:
        swa_started = True
        print(f"\nSWA activated at epoch {epoch + 1}")

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.training.epochs}")
    epoch_train_losses = {'loss': 0.0, 'loss_ci': 0.0, 'loss_mv': 0.0, 'loss_la': 0.0, 'loss_lb': 0.0, 'alpha': 0.0}
    n_batches = 0

    accum_steps = max(1, getattr(config.training, "gradient_accumulation_steps", 1))
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in enumerate(pbar):
        loss_fn.current_step = global_step
        volume = batch["volume"].to(device)
        targets = {
            "cell_indicator":   batch["cell_indicator"].to(device),
            "cell_mask":        batch["cell_mask"].to(device),
            "movement_vectors": batch["movement_vectors"].to(device),
        }

        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(volume, False)
            losses = loss_fn(outputs, targets)
            raw_loss = losses["loss"]
            scaled_loss = raw_loss / accum_steps

        scaler.scale(scaled_loss).backward()

        n_batches += 1
        is_accum_step = ((batch_idx + 1) % accum_steps == 0) or (batch_idx + 1 == len(train_loader))

        if is_accum_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            if swa_started:
                swa_model.update_parameters(model)

            global_step += 1

        for k in epoch_train_losses:
            if k in losses:
                epoch_train_losses[k] += float(losses[k])

        pbar.set_postfix(
            loss=f"{losses['loss']:.4f}",
            ci=f"{losses['loss_ci']:.4f}",
            mv=f"{losses['loss_mv']:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    # Step lr scheduler
    if swa_started:
        swa_scheduler.step()
    else:
        cosine_scheduler.step()

    # End of epoch calculations
    avg_train = {k: v / max(n_batches, 1) for k, v in epoch_train_losses.items()}
    val_metrics = validate(model, loss_fn, val_loader, device)
    val_loss = val_metrics["loss"]
    val_loss_ci = val_metrics["loss_ci"]
    val_loss_mv = val_metrics["loss_mv"]
    val_loss_la = val_metrics.get("loss_la", 0.0)
    val_loss_lb = val_metrics.get("loss_lb", 0.0)
    val_alpha = val_metrics.get("alpha", 0.0)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch + 1}: Train Loss = {avg_train['loss']:.4f} | Val Loss = {val_loss:.4f} (CI: {val_loss_ci:.4f}, MV: {val_loss_mv:.4f}, LA: {val_loss_la:.4f}, LB: {val_loss_lb:.4f}, Alpha: {val_alpha:.4f})")

    if hasattr(config.training, 'patience'):
        patience = config.training.patience
    else:
        patience = 0
        
    if val_loss < best_val_loss:
        val_checks_without_improvement = 0
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch + 1,
            'model': model.state_dict(),
            'val_loss': best_val_loss,
        }, best_ckpt_path)
        print(f"  ↑ New best checkpoint saved → {best_ckpt_path}")
    else:
        val_checks_without_improvement += 1
        print(f"  - No improvement for {val_checks_without_improvement} epochs")

    if patience > 0 and val_checks_without_improvement >= patience:
        print(f"Early stopping triggered.")
        break

    # Save last.pt after every epoch
    torch.save({
        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_checks_without_improvement': val_checks_without_improvement,
        'scheduler': cosine_scheduler.state_dict(),
        'swa_scheduler': swa_scheduler.state_dict(),
        'swa_started': swa_started,
        'swa_model': swa_model.state_dict(),
        'scaler': scaler.state_dict(),
    }, last_ckpt_path)
    print(f"  Saved last checkpoint to {last_ckpt_path}")

    # Update log history with all 14 metrics
    new_row = pd.DataFrame([{
        'epoch': epoch + 1,
        'train_loss': avg_train['loss'],
        'train_loss_ci': avg_train['loss_ci'],
        'train_loss_mv': avg_train['loss_mv'],
        'train_loss_la': avg_train['loss_la'],
        'train_loss_lb': avg_train['loss_lb'],
        'train_alpha': avg_train['alpha'],
        'val_loss': val_loss,
        'val_loss_ci': val_loss_ci,
        'val_loss_mv': val_loss_mv,
        'val_loss_la': val_loss_la,
        'val_loss_lb': val_loss_lb,
        'val_alpha': val_alpha,
        'lr': current_lr
    }])
    log_history = pd.concat([log_history, new_row], ignore_index=True)
    log_history.to_csv(log_csv_path, index=False)
    print(f"  Updated log file at {log_csv_path}")

    # Draw and display updated loss plot
    try:
        clear_output(wait=True)
        plot_loss_curves(log_history, plot_png_path)
    except Exception:
        pass

# ---- Post-training SWA updates ----
if swa_started:
    print("Updating SWA batch norms...")
    _update_swa_bn(swa_model, train_loader, device)
    
    # Safely extract the base model to save clean weights without DataParallel prefixes
    final_model = swa_model.module if hasattr(swa_model, 'module') else swa_model
    final_model = final_model.module if hasattr(final_model, 'module') else final_model
    
    torch.save({
        'model': final_model.state_dict(),
    }, OUTPUT_DIR / "swa_final.pt")
    print(f"Saved swa_final.pt to {OUTPUT_DIR}")

print("Training finished successfully!")


Beginning training run from Epoch 25 to 27...


Epoch 25/27:   0%|          | 0/3135 [00:00<?, ?it/s]

/tmp/ipykernel_58/1556800231.py:73: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  epoch_train_losses[k] += float(losses[k])


KeyboardInterrupt: 

In [ ]:
# import os
# import shutil
# import glob

# print("1. Searching for original Kaggle dataset...")
# search_paths = glob.glob('/kaggle/input/competitions/biohub-cell-tracking-during-development/train', recursive=True)
# data_dir = next((p for p in search_paths if 'biohub-cell-tracking-during-development' in p.lower()), None)

# if data_dir is None:
#     # Fallback to standard competition path
#     data_dir = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'

# print(f"   -> Using source: {data_dir}")
    
# OUTPUT_DIR = '/kaggle/working/mini_dataset/train'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Find all .zarr folders
# sample_ids = sorted(list(set(
#     f.split('.')[0] for f in os.listdir(data_dir) if f.endswith('.zarr')
# )))

# if not sample_ids:
#     print(f"❌ ERROR: Still couldn't find any .zarr folders inside {data_dir}")
# else:
#     samples_to_save = sample_ids[:1]
#     print(f"2. Copying {len(samples_to_save)} samples to {OUTPUT_DIR}...")
    
#     for i, sample_id in enumerate(samples_to_save):
#         zarr_src = os.path.join(data_dir, f"{sample_id}.zarr")
#         geff_src = os.path.join(data_dir, f"{sample_id}.geff")
        
#         zarr_dst = os.path.join(OUTPUT_DIR, f"{sample_id}.zarr")
#         geff_dst = os.path.join(OUTPUT_DIR, f"{sample_id}.geff")
        
#         if os.path.exists(zarr_src):
#             shutil.copytree(zarr_src, zarr_dst, dirs_exist_ok=True)
#         if os.path.exists(geff_src):
#             shutil.copytree(geff_src, geff_dst, dirs_exist_ok=True)

#     print(f"✅ Success! There are now {len(os.listdir(OUTPUT_DIR))} items in the mini_dataset/train folder.")
#     print("You can now run your dataloader cell safely!")